# Dynamic Task Scheduling Using SARSA-Based Reinforcement Learning
## Real Google Cluster Trace Data — Google Colab Notebook

This notebook builds a reproducible, tabular SARSA scheduler using a real shard of the Google Cluster Data 2011-2 trace.

**Important modelling note:** the trace's `task_events` table contains task submission requests, not actual task execution durations. This implementation therefore uses the CPU request as a **duration proxy** (short/long). The action `Wait / Defer` defers the current task for the current scheduling decision; it does not implement a persistent waiting queue. These assumptions must be stated in the project report.

### Workflow
1. Install/import libraries.
2. Download and load a real Google Cluster Trace shard.
3. Clean and convert submissions into task features.
4. Define the scheduling environment (state, actions, rewards, transitions).
5. Implement SARSA with correct terminal-state handling.
6. Train and plot the learning curve.
7. Compare SARSA, Round Robin and Random on exactly the same task window.
8. Run checks and launch a Gradio interface.

Run the cells from top to bottom in a fresh Colab runtime.

In [ ]:
!pip -q install gradio

In [ ]:
import os
import gzip
import shutil
import urllib.request
from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Libraries imported successfully.")

In [ ]:
# Google Cluster Data 2011-2: one task_events shard (compressed CSV)
TRACE_URL = (
    "https://storage.googleapis.com/clusterdata-2011-2/"
    "task_events/part-00000-of-00500.csv.gz"
)
DATA_DIR = Path("/content/google_cluster_trace")
DATA_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_GZ = DATA_DIR / "part-00000-of-00500.csv.gz"
LOCAL_CSV = DATA_DIR / "part-00000-of-00500.csv"

if not LOCAL_CSV.exists():
    print("Downloading Google Cluster Trace shard...")
    urllib.request.urlretrieve(TRACE_URL, LOCAL_GZ)
    print("Extracting compressed CSV...")
    with gzip.open(LOCAL_GZ, "rb") as source, open(LOCAL_CSV, "wb") as target:
        shutil.copyfileobj(source, target)
    print("Download and extraction completed.")
else:
    print("Dataset file already exists:", LOCAL_CSV)

print("CSV size (MB):", round(LOCAL_CSV.stat().st_size / (1024 * 1024), 2))

In [ ]:
# Column names from the Google Cluster Data 2011-2 task_events schema
COLUMN_NAMES = [
    "timestamp", "missing_info", "job_id", "task_index", "machine_id",
    "event_type", "user", "scheduling_class", "priority",
    "cpu_request", "memory_request", "disk_space_request",
    "different_machine_constraint"
]

# Load a bounded sample so the notebook remains practical in free Colab.
RAW_ROWS = 200_000
df = pd.read_csv(
    LOCAL_CSV,
    header=None,
    names=COLUMN_NAMES,
    nrows=RAW_ROWS,
    low_memory=False
)

print("Raw rows loaded:", len(df))
display(df.head())
print("\nColumns:", df.columns.tolist())

In [ ]:
# Keep task submission events (event_type == 0).
# Convert relevant columns to numeric safely before filtering.
for col in ["event_type", "priority", "cpu_request", "memory_request", "timestamp"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

submits = df.loc[df["event_type"] == 0].copy()
submits = submits.dropna(
    subset=["timestamp", "priority", "cpu_request", "memory_request"]
)
submits = submits.sort_values("timestamp").reset_index(drop=True)

if len(submits) < 100:
    raise ValueError(
        "Too few valid SUBMIT events were found. Check that the trace shard "
        "downloaded correctly and that the schema matches the file."
    )

# Google's production priorities are 9, 10 and 11.
submits["priority_level"] = (submits["priority"] >= 9).astype(int)

# The trace does not provide actual execution duration in task_events.
# Use CPU request as an explicitly stated proxy and split at its median.
median_cpu = float(submits["cpu_request"].median())
submits["duration_level"] = np.where(
    submits["cpu_request"] >= median_cpu, 2, 1
).astype(int)

task_stream = submits[
    ["timestamp", "priority_level", "duration_level", "cpu_request", "memory_request"]
].reset_index(drop=True)

print("Valid SUBMIT tasks:", len(task_stream))
print("Median CPU request:", median_cpu)
print("\nPriority distribution:")
display(
    task_stream["priority_level"]
    .map({0: "Low", 1: "High"})
    .value_counts()
    .rename_axis("Priority")
    .to_frame("Count")
)
print("\nDuration-proxy distribution:")
display(
    task_stream["duration_level"]
    .map({1: "Short", 2: "Long"})
    .value_counts()
    .rename_axis("Duration proxy")
    .to_frame("Count")
)
display(task_stream.head())

## Environment design

**State:** `(server_1_remaining, server_2_remaining, task_priority, task_duration)`.

- Each server's remaining busy time is 0, 1 or 2.
- Priority is 0 (Low) or 1 (High).
- Duration proxy is 1 (Short) or 2 (Long).
- Therefore the represented state space has at most `3 × 3 × 2 × 2 = 36` states.

**Actions:** 0 = assign to Server 1, 1 = assign to Server 2, 2 = wait/defer this task for this decision.

**Time transition:** after the decision, each positive server load decreases by one time unit. A task assigned with duration 2 remains busy for the next decision; duration 1 completes by the next decision.

**Important:** a task assigned to a busy server is rejected/deferred in this simplified simulator and receives a penalty. The simulator does not maintain a persistent queue. A future extension can add a real queue and queue-aware state.

In [ ]:
class TaskSchedulingEnv:
    """Small tabular scheduling simulator driven by a real task feature stream."""

    def __init__(self, tasks, num_servers=2, tasks_per_episode=20, seed=42):
        self.tasks = tasks.reset_index(drop=True)
        self.num_servers = int(num_servers)
        self.tasks_per_episode = int(tasks_per_episode)
        self.action_size = self.num_servers + 1
        self.rng = np.random.default_rng(seed)

        if self.num_servers != 2:
            raise ValueError("This project version is designed for exactly two servers.")
        if len(self.tasks) <= self.tasks_per_episode:
            raise ValueError("Task stream must contain more tasks than one episode.")

        self.reset()

    def _get_state(self):
        return tuple(self.server_remaining) + (
            int(self.current_priority),
            int(self.current_duration)
        )

    def _load_task(self):
        row = self.tasks.iloc[self.pointer]
        self.current_priority = int(row["priority_level"])
        self.current_duration = int(row["duration_level"])

    def reset(self, start_index=None):
        self.server_remaining = [0] * self.num_servers
        self.step_count = 0
        self.deferred_count = 0
        self.accepted_count = 0

        max_start = len(self.tasks) - self.tasks_per_episode
        if start_index is None:
            self.pointer = int(self.rng.integers(0, max_start + 1))
        else:
            self.pointer = int(start_index)
            if not 0 <= self.pointer <= max_start:
                raise ValueError("start_index is outside the valid task window.")

        self._load_task()
        return self._get_state()

    def step(self, action):
        if not isinstance(action, (int, np.integer)):
            raise TypeError("Action must be an integer.")
        if not 0 <= int(action) < self.action_size:
            raise ValueError("Action is outside the action space.")

        action = int(action)
        priority = self.current_priority
        duration = self.current_duration
        loads_before = self.server_remaining.copy()

        # Larger rewards/penalties for High-priority tasks.
        if action < self.num_servers:
            server = action
            if self.server_remaining[server] == 0:
                reward = 10 if priority == 1 else 5
                self.server_remaining[server] = duration
                decision = f"Assigned to Server {server + 1}"
                self.accepted_count += 1
            else:
                reward = -10 if priority == 1 else -5
                decision = f"Server {server + 1} busy — task deferred"
                self.deferred_count += 1
        else:
            if any(load == 0 for load in self.server_remaining):
                reward = -8 if priority == 1 else -3
                decision = "Wait / defer (free server available)"
            else:
                reward = 2
                decision = "Wait / defer (all servers busy)"
            self.deferred_count += 1

        # One scheduling time unit passes after this decision.
        self.server_remaining = [
            max(0, load - 1) for load in self.server_remaining
        ]

        self.step_count += 1
        done = self.step_count >= self.tasks_per_episode

        info = {
            "priority": "High" if priority == 1 else "Low",
            "duration": "Long" if duration == 2 else "Short",
            "loads_before": loads_before,
            "decision": decision,
            "reward": int(reward),
        }

        if done:
            next_state = None
        else:
            self.pointer += 1
            self._load_task()
            next_state = self._get_state()

        return next_state, int(reward), done, info


# Quick environment smoke test
test_env = TaskSchedulingEnv(task_stream, tasks_per_episode=5, seed=SEED)
test_state = test_env.reset(start_index=0)
print("Initial state:", test_state)
print("State length:", len(test_state))
print("Action count:", test_env.action_size)
print("One-step test:", test_env.step(0))

## SARSA agent

SARSA is an **on-policy temporal-difference control** algorithm. Its update uses the next action actually selected by the current behaviour policy.

For a non-terminal transition:

`Q(s,a) ← Q(s,a) + α [r + γ Q(s',a') − Q(s,a)]`

For a terminal transition, there is no next-state value:

`Q(s,a) ← Q(s,a) + α [r − Q(s,a)]`

The terminal case is handled explicitly in the implementation below.

In [ ]:
class SARSAAgent:
    def __init__(
        self,
        action_size,
        alpha=0.1,
        gamma=0.9,
        epsilon=0.3,
        min_epsilon=0.02,
        epsilon_decay=0.995,
        seed=42
    ):
        self.q_table = {}
        self.action_size = int(action_size)
        self.alpha = float(alpha)
        self.gamma = float(gamma)
        self.epsilon = float(epsilon)
        self.min_epsilon = float(min_epsilon)
        self.epsilon_decay = float(epsilon_decay)
        self.rng = np.random.default_rng(seed)

    def get_q(self, state, action):
        return float(self.q_table.get((tuple(state), int(action)), 0.0))

    def choose_action(self, state, greedy=False):
        if not greedy and self.rng.random() < self.epsilon:
            return int(self.rng.integers(self.action_size))

        values = [self.get_q(state, a) for a in range(self.action_size)]
        best_value = max(values)
        best_actions = [
            a for a, value in enumerate(values)
            if np.isclose(value, best_value)
        ]
        return int(self.rng.choice(best_actions))

    def update(self, state, action, reward, next_state=None, next_action=None, done=False):
        current_q = self.get_q(state, action)

        if done:
            # Terminal transition: no bootstrapped next-state value.
            target = float(reward)
        else:
            if next_state is None or next_action is None:
                raise ValueError("Non-terminal SARSA update requires next_state and next_action.")
            target = float(reward) + self.gamma * self.get_q(next_state, next_action)

        self.q_table[(tuple(state), int(action))] = (
            current_q + self.alpha * (target - current_q)
        )

    def decay_epsilon(self):
        self.epsilon = max(
            self.min_epsilon,
            self.epsilon * self.epsilon_decay
        )


print("SARSA agent class defined.")

In [ ]:
def train_sarsa(
    tasks,
    episodes=3000,
    tasks_per_episode=20,
    seed=42
):
    env = TaskSchedulingEnv(
        tasks,
        num_servers=2,
        tasks_per_episode=tasks_per_episode,
        seed=seed
    )
    agent = SARSAAgent(
        action_size=env.action_size,
        alpha=0.1,
        gamma=0.9,
        epsilon=0.3,
        min_epsilon=0.02,
        epsilon_decay=0.995,
        seed=seed
    )

    episode_rewards = []

    for episode in range(episodes):
        state = env.reset()
        action = agent.choose_action(state)
        total_reward = 0

        while True:
            next_state, reward, done, info = env.step(action)
            total_reward += reward

            if done:
                agent.update(
                    state, action, reward,
                    next_state=None, next_action=None, done=True
                )
                break

            next_action = agent.choose_action(next_state)
            agent.update(
                state, action, reward,
                next_state=next_state,
                next_action=next_action,
                done=False
            )
            state, action = next_state, next_action

        agent.decay_epsilon()
        episode_rewards.append(total_reward)

    return agent, env, episode_rewards


NUM_SERVERS = 2
EPISODES = 3000
TASKS_PER_EPISODE = 20

trained_agent, training_env, episode_rewards = train_sarsa(
    task_stream,
    episodes=EPISODES,
    tasks_per_episode=TASKS_PER_EPISODE,
    seed=SEED
)

print("Training completed.")
print("Episodes:", len(episode_rewards))
print("Final epsilon:", round(trained_agent.epsilon, 4))
print("Visited state-action pairs:", len(trained_agent.q_table))
print("Mean reward, first 100 episodes:", round(np.mean(episode_rewards[:100]), 2))
print("Mean reward, last 100 episodes:", round(np.mean(episode_rewards[-100:]), 2))

In [ ]:
# Plot raw episode reward and 50-episode moving average.
rolling_reward = pd.Series(episode_rewards).rolling(
    window=50, min_periods=1
).mean()

plt.figure(figsize=(10, 5))
plt.plot(episode_rewards, alpha=0.25, label="Episode reward")
plt.plot(rolling_reward, linewidth=2, label="50-episode moving average")
plt.xlabel("Episode")
plt.ylabel("Total reward")
plt.title("SARSA Training Progress — Google Cluster Trace")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

print("Training curve generated.")

## Fair evaluation against baseline policies

All three policies are evaluated on the **same contiguous task window** selected from a fixed seed. This avoids comparing policies on different task sequences.

- **SARSA:** trained Q-table, greedy action selection (no exploration).
- **Round Robin:** alternates Server 1 and Server 2.
- **Random:** randomly selects Server 1, Server 2 or Wait/Defer.

The reported reward is the reward defined by this simulator; it is not a direct measurement of real-world job latency or throughput.

In [ ]:
def run_policy(policy_name, tasks, num_tasks=20, seed=42, start_index=None):
    env = TaskSchedulingEnv(
        tasks,
        num_servers=2,
        tasks_per_episode=num_tasks,
        seed=seed
    )

    if start_index is None:
        max_start = len(tasks) - num_tasks
        start_index = seed % (max_start + 1)

    state = env.reset(start_index=start_index)
    rng = np.random.default_rng(seed)
    rows = []
    total_reward = 0
    rr_pointer = 0

    for task_number in range(1, num_tasks + 1):
        if policy_name == "SARSA":
            action = trained_agent.choose_action(state, greedy=True)
        elif policy_name == "Round Robin":
            action = rr_pointer % env.num_servers
            rr_pointer += 1
        elif policy_name == "Random":
            action = int(rng.integers(env.action_size))
        else:
            raise ValueError("Unknown policy: " + policy_name)

        next_state, reward, done, info = env.step(action)
        rows.append({
            "Task #": task_number,
            "Priority": info["priority"],
            "Duration proxy": info["duration"],
            "Server loads before decision": str(info["loads_before"]),
            "Decision": info["decision"],
            "Reward": reward
        })
        total_reward += reward

        if done:
            break
        state = next_state

    result = {
        "Policy": policy_name,
        "Total reward": int(total_reward),
        "Accepted tasks": int(env.accepted_count),
        "Deferred tasks": int(env.deferred_count)
    }
    return pd.DataFrame(rows), result


def evaluate_all(num_tasks=30, seed=42):
    num_tasks = int(num_tasks)
    seed = int(seed)
    if not 5 <= num_tasks <= 100:
        raise ValueError("Choose between 5 and 100 tasks.")

    max_start = len(task_stream) - num_tasks
    start_index = seed % (max_start + 1)

    logs = {}
    summaries = []
    for name in ["SARSA", "Round Robin", "Random"]:
        log, summary = run_policy(
            name, task_stream,
            num_tasks=num_tasks,
            seed=seed,
            start_index=start_index
        )
        logs[name] = log
        summaries.append(summary)

    summary_df = pd.DataFrame(summaries)
    return logs, summary_df


evaluation_logs, comparison_df = evaluate_all(num_tasks=30, seed=42)
display(comparison_df)
print("\nSARSA decisions:")
display(evaluation_logs["SARSA"].head(10))

In [ ]:
# Visual comparison of total reward
plt.figure(figsize=(8, 5))
bars = plt.bar(
    comparison_df["Policy"],
    comparison_df["Total reward"]
)
plt.ylabel("Total reward")
plt.title("Scheduling Policy Comparison on the Same Google Trace Task Window")
plt.grid(axis="y", alpha=0.25)

for bar, value in zip(bars, comparison_df["Total reward"]):
    offset = 2 if value >= 0 else -2
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + offset,
        str(value),
        ha="center",
        va="bottom" if value >= 0 else "top"
    )

plt.tight_layout()
plt.show()

sarsa_reward = int(
    comparison_df.loc[
        comparison_df["Policy"] == "SARSA", "Total reward"
    ].iloc[0]
)
rr_reward = int(
    comparison_df.loc[
        comparison_df["Policy"] == "Round Robin", "Total reward"
    ].iloc[0]
)
print(f"SARSA minus Round Robin reward: {sarsa_reward - rr_reward:+d}")

## Basic validation tests

These checks verify the state/action dimensions, reward rules, terminal SARSA update and reproducibility of the fixed evaluation window. They do not replace a full software test suite.

In [ ]:
# Test 1: state has two server loads + priority + duration.
env_test = TaskSchedulingEnv(task_stream, tasks_per_episode=5, seed=SEED)
s = env_test.reset(start_index=0)
assert len(s) == 4
assert env_test.action_size == 3
assert all(load in (0, 1, 2) for load in s[:2])
assert s[2] in (0, 1) and s[3] in (1, 2)

# Test 2: assigning a High-priority task to a free server earns +10.
env_test.reset(start_index=0)
env_test.current_priority = 1
env_test.current_duration = 2
env_test.server_remaining = [0, 0]
_, reward, _, _ = env_test.step(0)
assert reward == 10

# Test 3: assigning to a busy server incurs the High-priority penalty.
env_test.reset(start_index=0)
env_test.current_priority = 1
env_test.current_duration = 1
env_test.server_remaining = [2, 0]
_, reward, _, _ = env_test.step(0)
assert reward == -10

# Test 4: terminal SARSA update uses reward only.
agent_test = SARSAAgent(action_size=3, alpha=0.1, gamma=0.9, seed=SEED)
state_test = (0, 0, 1, 2)
agent_test.update(state_test, 0, 10, done=True)
assert np.isclose(agent_test.get_q(state_test, 0), 1.0)

# Test 5: same policy and fixed start window are reproducible.
log_a, result_a = run_policy(
    "Round Robin", task_stream, num_tasks=10, seed=7, start_index=7
)
log_b, result_b = run_policy(
    "Round Robin", task_stream, num_tasks=10, seed=7, start_index=7
)
assert result_a == result_b
assert log_a.equals(log_b)

print("All basic validation tests PASSED.")

## Gradio interactive demo

The app displays the SARSA decisions and compares total rewards for the same task window. The reward chart reflects the simulator's designed reward function, not measured production-cluster performance.

In [ ]:
def gradio_simulation(num_tasks, seed):
    logs, summary_df = evaluate_all(
        num_tasks=int(num_tasks),
        seed=int(seed)
    )

    summary_text = summary_df.to_markdown(index=False)
    summary_text += (
        "\n\n**Interpretation:** Higher total reward means a policy "
        "performed better under this project's reward design for this "
        "specific task window. Repeat with multiple seeds before drawing "
        "a general conclusion."
    )

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(summary_df["Policy"], summary_df["Total reward"])
    ax.set_ylabel("Total reward")
    ax.set_title("Policy comparison on the same task sequence")
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()

    return logs["SARSA"], summary_text, fig


demo = gr.Interface(
    fn=gradio_simulation,
    inputs=[
        gr.Slider(
            minimum=5, maximum=100, value=30, step=1,
            label="Number of tasks"
        ),
        gr.Number(value=42, precision=0, label="Evaluation seed")
    ],
    outputs=[
        gr.Dataframe(label="SARSA step-by-step decisions"),
        gr.Markdown(label="Reward comparison"),
        gr.Plot(label="Policy reward chart")
    ],
    title="Dynamic Task Scheduling Using SARSA",
    description=(
        "A tabular SARSA scheduler trained using real Google Cluster "
        "Trace task submission records."
    )
)

demo.launch(share=True)

## Report-writing notes and limitations

Use the actual values printed by your own run in the report. Do not copy example values from another run.

### What this project demonstrates
- Loading and preprocessing real Google Cluster Trace task submission events.
- Formulating a finite scheduling simulation as a tabular reinforcement-learning problem.
- Training an on-policy SARSA agent with epsilon-greedy exploration.
- Comparing the trained policy with Round Robin and Random baselines on identical task windows.
- Displaying decisions and reward comparisons through Gradio.

### Limitations to disclose
1. CPU request is a proxy for task duration; it is not observed runtime.
2. The simulator has two identical servers.
3. Wait/Defer does not maintain a persistent waiting queue.
4. The reward values are manually designed; conclusions are specific to that reward function.
5. This is a trace-driven simulation, not a live deployment on Google's production cluster.
6. A stronger evaluation should report averages and variation over many fixed seeds, plus queueing delay, task completion time, utilisation and fairness once those are modelled.

### Dataset reference
Google Cluster Data 2011-2, `task_events` table, shard `part-00000-of-00500.csv.gz`. The project uses the public trace as downloaded from Google's Cluster Data storage. Cite the dataset and its applicable terms/licence in the final report.

**Submission evidence to capture:** dataset statistics output, learning curve, comparison table and chart, passing validation message, and a screenshot of the running Gradio app after clicking its submit/run button.